# 🦠 COVID-19 Data Analysis Project
## Notebook 2: Exploratory Data Analysis (EDA)

**Author:** Ahmed  
**Date:** March 2026

---

### 📌 Objectives
1. Load the `cleaned_covid.csv` data
2. Perform **Descriptive Statistics** (mean, median, variance)
3. Understand **Distributions** of Cases, Deaths, and Recoveries
4. Analyze **Correlations** between numerical features
5. Provide Continent/Regional statistical summaries

---

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set visualization styles for professional look
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
sns.set_context("notebook", font_scale=1.1)

print('✅ Libraries imported successfully! Visual styles applied.')

---
## 📂 Step 2 — Load the Cleaned Data

In [ ]:
# Read from the processed folder
df = pd.read_csv('../data/processed/cleaned_covid.csv')

print(f'Data Loaded: {df.shape[0]} Countries × {df.shape[1]} Features\n')
df.head()

---
## 📊 Step 3 — Descriptive Statistics
Let's look at the central tendency and dispersion of our key metrics.

In [ ]:
# Key metrics for analysis
key_metrics = ['TotalCases', 'TotalDeaths', 'TotalRecovered', 'TotalTests', 
               'CaseFatalityRate', 'RecoveryRate', 'TestsPerCase']

# Using .describe() to get count, mean, std, min, max, quartiles
summary_stats = df[key_metrics].describe().T
summary_stats['median'] = df[key_metrics].median()

# Reorder columns for better reading
summary_stats = summary_stats[['count', 'mean', 'median', 'std', 'min', 'max']]
summary_stats.round(2)

### 💡 Key Insights from Statistics:
- **High Variance**: Metrics like `TotalCases` and `TotalDeaths` have a standard deviation much higher than their mean. This indicates extreme **skewness** — a few countries have massive outbreaks, while many have very few cases.
- **Testing Rate**: It varies wildly. Looking at `TestsPerCase`, some countries test heavily while others barely test.

---
## 📈 Step 4 — Distribution Analysis
Because of the extreme skewness, we will use **Logarithmic Scale** for `TotalCases` and `TotalDeaths` to visualize their distributions properly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of Total Cases (Log Scale)
sns.histplot(df['TotalCases'], bins=30, kde=True, log_scale=True, ax=axes[0], color='blue')
axes[0].set_title('Distribution of Total Cases (Log10 Scale)')
axes[0].set_xlabel('Total Cases')
axes[0].set_ylabel('Frequency')

# Distribution of Total Deaths (Log Scale, adding +1 to avoid log(0))
sns.histplot(df['TotalDeaths'] + 1, bins=30, kde=True, log_scale=True, ax=axes[1], color='red')
axes[1].set_title('Distribution of Total Deaths (Log10 Scale)')
axes[1].set_xlabel('Total Deaths (+1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

Let's look at the distribution of our **calculated rates** (`CaseFatalityRate` and `RecoveryRate`), which are percentages and don't need a log scale.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Case Fatality Rate
sns.histplot(df['CaseFatalityRate'], bins=20, kde=True, ax=axes[0], color='darkred')
axes[0].set_title('Distribution of Case Fatality Rate (%)')
axes[0].set_xlabel('CFR (%)')
axes[0].axvline(df['CaseFatalityRate'].median(), color='black', linestyle='--', label=f'Median: {df["CaseFatalityRate"].median():.2f}%')
axes[0].legend()

# Recovery Rate
sns.histplot(df['RecoveryRate'], bins=20, kde=True, ax=axes[1], color='green')
axes[1].set_title('Distribution of Recovery Rate (%)')
axes[1].set_xlabel('Recovery Rate (%)')
axes[1].axvline(df['RecoveryRate'].median(), color='black', linestyle='--', label=f'Median: {df["RecoveryRate"].median():.2f}%')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 🔗 Step 5 — Correlation Analysis
How do different metrics relate to each other? For example, does testing more lead to a lower fatality rate?

In [ ]:
# Select only numeric columns for correlation
numeric_df = df.select_dtypes(include=[np.number])

# Variables of interest
corr_cols = ['Population', 'TotalCases', 'TotalDeaths', 'TotalRecovered', 
             'TotalTests', 'TestsPer1M', 'CaseFatalityRate']

corr_matrix = numeric_df[corr_cols].corr(method='spearman') # Spearman is better for skewed data

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title('Spearman Correlation Heatmap (Key Features)', pad=15, size=14)
plt.show()

### 💡 Correlation Insights:
- **TotalCases & TotalDeaths** are highly positively correlated (close to 1), which is expected.
- **TotalTests & TotalCases** are also very highly correlated. More testing = more confirmed cases.
- **Case Fatality Rate** shows very weak correlation with most metrics.

---
## 🌍 Step 6 — Regional / Continental Aggregation
Let's see the macro-level impact by comparing Continents and WHO Regions.

In [ ]:
# Group by Continent
continent_stats = df.groupby('Continent').agg({
    'Country': 'count',
    'TotalCases': 'sum',
    'TotalDeaths': 'sum',
    'TotalRecovered': 'sum',
}).rename(columns={'Country': 'Num_Countries'})

# Calculate Aggregated Rates
continent_stats['CFR (%)'] = (continent_stats['TotalDeaths'] / continent_stats['TotalCases'] * 100).round(2)
continent_stats['Recovery (%)'] = (continent_stats['TotalRecovered'] / continent_stats['TotalCases'] * 100).round(2)

continent_stats.sort_values('TotalCases', ascending=False)

In [ ]:
# Group by WHO Region
who_stats = df.groupby('WHORegion').agg({
    'Country': 'count',
    'TotalCases': 'sum',
    'TotalDeaths': 'sum',
}).rename(columns={'Country': 'Num_Countries'})

who_stats.sort_values('TotalCases', ascending=False)

---
## 🎉 EDA Complete!
We now thoroughly understand our data distribution, correlations, and general statistics. 

**Next up: Notebook 3 — Advanced Visualizations!**